# Model 1: Simple Baseline
This notebook builds a simple baseline model to predict smartphone addiction.
We will use Logistic Regression.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
# Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
# Basic exploration
print("Train shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
# Separate features and target
X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [ ]:
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [ ]:
# Define the model
model = LogisticRegression(random_state=42, max_iter=1000)

In [ ]:
# Create and evaluate the pipeline
clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', model)])

# Train the model
clf.fit(X, y)
print("Model trained.")

In [ ]:
# Make predictions on test set
preds = clf.predict_proba(X_test)[:, 1]

In [ ]:
# Create submission file
submission = pd.DataFrame({'id': test['id'], 'addicted_label': preds})
submission.to_csv('submission_1.csv', index=False)
print("Submission saved to submission_1.csv")